## Step 1: Retrieve the HTML of the Target Web Page

The first step involves programmatically loading the target web page and retrieving its HTML content. This notebook uses `requests` and `lxml` for static pages such as Wikipedia and Alukah.

Two example URLs are used:

1. Alukah article: `https://www.alukah.net/culture/0/180954/`
2. Wikipedia article: `https://ar.wikipedia.org/wiki/فروع_الكيمياء`


In [6]:
import requests
from lxml import html

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

# Replace with your article URL
# Example URLs
url = "https://www.alukah.net/culture/0/180954/"
url = "https://ar.wikipedia.org/wiki/فروع_الكيمياء"

# Fetch the page
response = requests.get(url, headers=headers)

print("Status:", response.status_code)
print("HTML length:", len(response.text))

# Parse HTML
html_text = response.text
tree = html.fromstring(html_text)

print("Tree:", tree)
print("Tree tag:", tree.tag)

Status: 200
HTML length: 139003
Tree: <Element html at 0x10434b890>
Tree tag: html


In [7]:
print(html_text[:1000])

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" lang="ar" dir="rtl">
<head>
<meta charset="UTF-8">
<title>فروع الكيمياء - ويكيبيديا</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector

## Step 2: Generate XPath Expressions Using ChatGPT

Once the HTML is obtained, specific snippets containing the desired data (e.g., title, author, content, views, date) are extracted. These snippets are then fed to ChatGPT with a prompt to generate robust XPath expressions:

### Model Note

This notebook follows the same XPath-generation prompt style that was previously tested with the **ChatGPT-4o model in 2024**. The goal is to keep the prompt reusable across websites while validating the generated XPath expressions with real extraction examples.

### Role-Based Prompt Engineering for XPath Generation

“Prompt Engineering Strategy for Automated XPath Generation”

To automate XPath generation, ChatGPT is assigned a structured role. The following system prompt defines its behavior:

```text
You are a professional software engineer specialized in web scraping and HTML parsing.

Your task is to analyze the HTML code provided by the user and generate robust XPath expressions that can extract the requested target values.

Instructions:

1. Carefully read the HTML snippet provided by the user.
2. Identify the structural patterns surrounding the target value.
3. Generate XPath expressions that reliably locate the target element.
4. Avoid hardcoding exact text values appearing in the HTML.
5. When text-based predicates are necessary, always use:
   contains(., 'value')
   instead of:
   text()='value'
6. Prefer:
   contains(@class, '...')
   contains(@id, '...')
   rather than strict equality.
7. Ensure the XPath is robust against:
   - Additional classes being added
   - Minor ID changes
   - Extra wrapper elements
8. Output only the XPath expressions, clearly labeled by target field.

HTML snippet:
```

---

### Target field example:

1. title
2. author
3. article
4. date

---

### HTML snippet example:

```html
<div id="content">
  <h1 id="firstHeading" class="firstHeading">فروع الكيمياء</h1>
  <div class="mw-parser-output">
    <p>
      فروع أو (أقسام الكيمياء) عديدة فعلم الكيمياء له تأثير على معظم نواحي
      الحياة.
    </p>
    <p>
      فالتفاعلات والتحولات الكيميائية تلعب دوراً أساسياً في المجالات العلمية
      المختلفة.
    </p>
  </div>
  <li id="footer-info-lastmod">
    تم التعديل آخر مرة في 24 مارس 2025، الساعة 07:24
  </li>
</div>
```

---

### Example of XPaths Generated

```xpath
1. Title: //h1[contains(@id,'firstHeading')]//text()
2. Article: //div[contains(@class,'mw-parser-output')]//p//text()
3. Last Modified Date: //li[contains(@id,'footer-info-lastmod')]//text()
```

---


In [8]:
SYSTEM_PROMPT = f"""
You are a professional software engineer specialized in web scraping and HTML parsing.

Your task is to analyze the HTML code provided by the user and generate robust XPath expressions that can extract the requested target values.

Instructions:

1. Carefully read the HTML snippet provided by the user.
2. Identify the structural patterns surrounding the target value.
3. Generate XPath expressions that reliably locate the target element.
4. Avoid hardcoding exact text values appearing in the HTML.
5. When text-based predicates are necessary, always use:
   contains(., 'value')
   instead of:
   text()='value'
6. Prefer:
   contains(@class, '...')
   contains(@id, '...')
   rather than strict equality.
7. Ensure the XPath is robust against:
   - Additional classes being added
   - Minor ID changes
   - Extra wrapper elements
8. Output only the XPath expressions, clearly labeled by target field.

HTML snippet:
{html_text}
"""

print(SYSTEM_PROMPT)


You are a professional software engineer specialized in web scraping and HTML parsing.

Your task is to analyze the HTML code provided by the user and generate robust XPath expressions that can extract the requested target values.

Instructions:

1. Carefully read the HTML snippet provided by the user.
2. Identify the structural patterns surrounding the target value.
3. Generate XPath expressions that reliably locate the target element.
4. Avoid hardcoding exact text values appearing in the HTML.
5. When text-based predicates are necessary, always use:
   contains(., 'value')
   instead of:
   text()='value'
6. Prefer:
   contains(@class, '...')
   contains(@id, '...')
   rather than strict equality.
7. Ensure the XPath is robust against:
   - Additional classes being added
   - Minor ID changes
   - Extra wrapper elements
8. Output only the XPath expressions, clearly labeled by target field.

HTML snippet:
<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-ena

## Step 3: Test and Validate the Generated XPath

The XPath expressions returned by ChatGPT are tested using Python libraries such as Scrapy-Playwright or lxml to ensure they correctly extract the intended elements from the HTML

This step confirms that the generated XPath expressions reliably extract data even if the underlying HTML structure has minor changes.

This notebook validates the extraction logic with **two examples**:

1. **Alukah article page**
2. **Wikipedia article page**

The same validation idea can be reused for any URL by adding a new XPath rule set for that website. The test checks that required fields such as title, URL, and content are extracted successfully.


In [9]:
from lxml import html
from playwright.async_api import async_playwright

# 1. XPath configs generated by GPT

ALUKAH_SELECTORS = {
    "site": "alukah",
    "title_xpath": "//h1[contains(@class,'larger2Font')]//span[@itemprop='name']/text()",
    "author_xpath": "//meta[@property='article:author']/@content",
    "author_name_xpath": "//span[contains(@id,'lblAuthor')]//a//span[@itemprop='name']/text()",
    "author_name_fallback_xpath": "//span[contains(@class,'latest-authorName')]//a/text()",
    "content_xpath": (
        "//div[@id='print_area']"
        "//div[contains(@class,'content-area')]"
        "//*[contains(@id,'ArticleContent')]//text()"
    ),
    "content_fallback_xpath": "//div[contains(@id,'ArticleContent')]//text()",
    "article_url_xpath": "//link[@rel='canonical']/@href",
    "published_date_xpath": (
        "//span[contains(@id,'lblPublishDate')]"
        "//meta[@itemprop='datePublished']/@content"
    ),
    "views_xpath": "//span[contains(@id,'lblVisitCount')]/text()",
}


WIKIPEDIA_SELECTORS = {
    "site": "wikipedia",
    "title_xpath": "//h1[contains(@id,'firstHeading')]//text()",
    "content_xpath": "//div[contains(@class,'mw-parser-output')]//p//text()",
    "article_url_xpath": "//link[@rel='canonical']/@href",
    "last_modified_xpath": "//li[contains(@id,'footer-info-lastmod')]/text()",
}

In [10]:
# ============================================================
# 2. Helper functions
# ============================================================

def clean_text(value):
    """
    This function removes leading and trailing whitespace.
    If the value is not a valid non-empty string, it returns None.
    """
    return value.strip() if isinstance(value, str) and value.strip() else None


def extract_first(tree, xpath_expr):
    """
    Extract the first matching value from an XPath expression.

    Parameters:
        tree:
            Parsed HTML tree created by lxml.html.fromstring().
        xpath_expr:
            XPath selector generated by the LLM or written manually.

    Returns:
        The first matched text/attribute value as a cleaned string.
        Returns None if the XPath is missing or no result is found.
    """
    # If no XPath selector is provided, return None
    if not xpath_expr:
        return None

    # Run XPath query on the parsed HTML tree
    result = tree.xpath(xpath_expr)

    # If XPath does not match anything, return None
    if not result:
        return None

    # Convert the first result to string and clean it
    return clean_text(str(result[0]))


def extract_joined(tree, xpath_expr):
    """
    Extract all matching values from an XPath expression and join them.

    This is useful for article content because content usually appears
    across many text nodes or paragraphs.

    Parameters:
        tree:
            Parsed HTML tree created by lxml.html.fromstring().
        xpath_expr:
            XPath selector generated by the LLM or written manually.

    Returns:
        A single string containing all matched text values joined by spaces.
        Returns an empty string if the XPath is missing or no result is found.
    """
    # If no XPath selector is provided, return an empty string
    if not xpath_expr:
        return ""

    # Run XPath query and collect all matching nodes/text values
    result = tree.xpath(xpath_expr)

    # If XPath does not match anything, return an empty string
    if not result:
        return ""

    # Clean each matched value and join all non-empty values into one text
    return " ".join(
        str(r).strip()
        for r in result
        if str(r).strip()
    )


async def fetch_html_with_playwright(url):
    """
    Fetch the fully rendered HTML of a webpage using Playwright.

    Playwright is used instead of requests because some websites load
    parts of their page using JavaScript. This function opens the page
    in a headless Chromium browser, waits briefly for the DOM to load,
    and then returns the rendered HTML.

    Parameters:
        url:
            The webpage URL to scrape.

    Returns:
        A dictionary containing:
            - status: HTTP response status code
            - html: rendered page HTML
    """
    # Start Playwright
    async with async_playwright() as p:

        # Launch Chromium in headless mode, meaning no browser window is shown
        browser = await p.chromium.launch(headless=True)

        # Create a new browser page with a normal browser User-Agent
        # This helps avoid some simple bot-blocking behavior
        page = await browser.new_page(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            )
        )

        # Open the URL and wait until the DOM content is loaded
        # domcontentloaded is usually faster and more reliable than networkidle
        response = await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=60000
        )

        # Wait a short time to allow JavaScript-rendered elements to appear
        await page.wait_for_timeout(3000)

        # Get the rendered HTML after the page has loaded
        page_html = await page.content()

        # Close browser to free resources
        await browser.close()

    # Return both HTTP status and rendered HTML
    return {
        "status": response.status if response else None,
        "html": page_html,
    }

In [11]:
# 3. Generic extractor

async def scrape_with_selectors(url, selectors):
    """
    Scrape a webpage using a dictionary of XPath selectors.

    This function is generic, meaning it can work with different websites
    as long as we provide the correct selector dictionary for that site.

    Parameters:
        url:
            The webpage URL to scrape.
        selectors:
            A dictionary containing XPath expressions for fields such as:
            title, content, article_url, author, published_date, views, etc.

    Returns:
        item:
            A dictionary containing the extracted data.
    """

    # Fetch the rendered HTML using Playwright
    # This allows the scraper to handle pages that load content with JavaScript
    fetched = await fetch_html_with_playwright(url)

    # Print basic debugging information about the fetched page
    print("URL:", url)
    print("HTTP status:", fetched["status"])
    print("HTML length:", len(fetched["html"]))

    # Parse the rendered HTML into an lxml tree
    # This tree allows us to apply XPath expressions
    tree = html.fromstring(fetched["html"])

    # Create an empty dictionary to store extracted fields
    item = {}

    # Store metadata about the website and original URL
    item["site"] = selectors.get("site")
    item["source_url"] = url

    # Extract the article title using the title XPath
    item["title"] = extract_first(
        tree,
        selectors.get("title_xpath")
    )

    # Extract the main article content
    # This usually joins many text nodes into one text string
    item["content"] = extract_joined(
        tree,
        selectors.get("content_xpath")
    )

    # If the main content XPath fails, try a fallback content XPath
    # This makes the extractor more robust for pages with slightly different HTML
    if not item["content"] and selectors.get("content_fallback_xpath"):
        item["content"] = extract_joined(
            tree,
            selectors.get("content_fallback_xpath")
        )

    # Extract canonical article URL
    item["article_url"] = extract_first(
        tree,
        selectors.get("article_url_xpath")
    )

    # ------------------------------------------------------------
    # Optional fields
    # These fields may exist on some websites but not others.
    # The function only extracts them if the selector dictionary includes them.

    # Extract author metadata if available
    if selectors.get("author_xpath"):
        item["author"] = extract_first(
            tree,
            selectors.get("author_xpath")
        )

    # Extract displayed author name if available
    if selectors.get("author_name_xpath"):
        item["author_name"] = extract_first(
            tree,
            selectors.get("author_name_xpath")
        )

        # If the main author name selector fails, try a fallback selector
        if not item["author_name"] and selectors.get("author_name_fallback_xpath"):
            item["author_name"] = extract_first(
                tree,
                selectors.get("author_name_fallback_xpath")
            )

    # Extract published date if the website provides it
    if selectors.get("published_date_xpath"):
        item["published_date"] = extract_first(
            tree,
            selectors.get("published_date_xpath")
        )

    # Extract view count if the website provides it
    if selectors.get("views_xpath"):
        item["views"] = extract_first(
            tree,
            selectors.get("views_xpath")
        )

    # Extract last-modified date if the website provides it
    # Example: Wikipedia footer information
    if selectors.get("last_modified_xpath"):
        item["last_modified"] = extract_first(
            tree,
            selectors.get("last_modified_xpath")
        )
    # ------------------------------------------------------------

    # Print extracted result for quick notebook inspection
    print("\n" + "=" * 80)
    print("Site:", item.get("site"))
    print("Title:", item.get("title"))
    print("Content:", item.get("content", "")[:110], "...")
    print("Article URL:", item.get("article_url"))

    # Print optional fields only if they were extracted successfully
    if item.get("author"):
        print("Author:", item.get("author"))

    if item.get("author_name"):
        print("Author Name:", item.get("author_name"))

    if item.get("published_date"):
        print("Published Date:", item.get("published_date"))

    if item.get("last_modified"):
        print("Last Modified:", item.get("last_modified"))

    if item.get("views"):
        print("Views:", item.get("views"))

    print("=" * 80)

    # Return extracted data as a Python dictionary
    return item

In [12]:
# 4. Validation test

def validate_item(item):
    """
    Validate the extracted item.

    This function checks whether the most important required fields
    were successfully extracted from the webpage.

    Required fields:
        - title
        - content
        - article_url

    Parameters:
        item:
            A dictionary returned by scrape_with_selectors().

    Returns:
        result:
            A dictionary showing whether validation passed,
            which fields are missing, and basic length statistics.
    """

    # Define the fields that must exist for extraction to be considered successful
    required_fields = ["title", "content", "article_url"]

    # Check which required fields are missing or empty
    missing = [
        field for field in required_fields
        if not item.get(field)
    ]

    # Store validation result and useful debugging statistics
    result = {
        # Original webpage URL
        "url": item.get("source_url"),

        # Website/source name, such as "alukah" or "wikipedia"
        "site": item.get("site"),

        # Passes only if no required fields are missing
        "passed": len(missing) == 0,

        # List of required fields that failed extraction
        "missing_fields": missing,

        # Length of extracted title, useful for detecting empty/invalid extraction
        "title_length": len(item.get("title") or ""),

        # Length of extracted content, useful for detecting weak/partial extraction
        "content_length": len(item.get("content") or ""),
    }

    # Return validation summary
    return result


async def run_validation_tests():
    """
    Run validation tests on multiple example websites.

    This function tests the cached XPath selectors on representative URLs.
    It helps verify that the LLM-generated selectors still work and that
    required fields are extracted correctly.

    Current test cases:
        1. Alukah article page
        2. Wikipedia article page

    Returns:
        results:
            A list of validation dictionaries, one for each tested URL.
    """

    # Define test cases.
    # Each test contains a URL and the selector dictionary for that website.
    tests = [
        {
            "url": "https://www.alukah.net/culture/0/180954/",
            "selectors": ALUKAH_SELECTORS,
        },
        {
            "url": "https://en.wikipedia.org/wiki/Python_(programming_language)",
            "selectors": WIKIPEDIA_SELECTORS,
        },
    ]

    # Store validation results for all test cases
    results = []

    # Loop through each test case
    for test in tests:

        # Scrape the page using the corresponding XPath selector dictionary
        item = await scrape_with_selectors(
            test["url"],
            test["selectors"]
        )

        # Validate the extracted item
        validation = validate_item(item)

        # Save validation result
        results.append(validation)

        # Print validation result for notebook inspection
        print("\nValidation:")
        print(validation)
        print("\n")

    # Return all validation results
    return results

In [13]:
validation_results = await run_validation_tests()
validation_results

URL: https://www.alukah.net/culture/0/180954/
HTTP status: 200
HTML length: 152832

Site: alukah
Title: عفوا: سعادتك تم اختراقها وسرقتها
Content: عفوًا: سعادتك تم اختراقها وسرقتها!! لقد أحدثت منصات الإنترنت ثورة في حياتنا، لكننا الآن فقط بدأنا نرى الجانب ا ...
Article URL: https://www.alukah.net/culture/0/180954/عفوا-سعادتك-تم-اختراقها-وسرقتها/
Author: http://www.alukah.net/authors/view/sharia/2846/
Author Name: د. زيد بن محمد الرماني
Published Date: 14/2/2026
Views: 2963

Validation:
{'url': 'https://www.alukah.net/culture/0/180954/', 'site': 'alukah', 'passed': True, 'missing_fields': [], 'title_length': 32, 'content_length': 7513}


URL: https://en.wikipedia.org/wiki/Python_(programming_language)
HTTP status: 200
HTML length: 842274

Site: wikipedia
Title: Python (programming language)
Content: Python is a high-level , general-purpose programming language that emphasizes code readability , simplicity, a ...
Article URL: https://en.wikipedia.org/wiki/Python_(programming_language)
Las

[{'url': 'https://www.alukah.net/culture/0/180954/',
  'site': 'alukah',
  'passed': True,
  'missing_fields': [],
  'title_length': 32,
  'content_length': 7513},
 {'url': 'https://en.wikipedia.org/wiki/Python_(programming_language)',
  'site': 'wikipedia',
  'passed': True,
  'missing_fields': [],
  'title_length': 29,
  'content_length': 23227}]

## Step 4: Automate the Extraction Workflow

Finally, the validated XPath expressions are integrated into a Scrapy-Playwright pipeline to automate the data extraction process for multiple web pages. This ensures consistency in collecting structured web data.
